# FRED Data Exploration
I exployed the data to understand the data structure, to check what cleaning steps are required for this data. I looked for the data type, date type, null values etc. 

## Setup
import the libraries used for the data exploration process 

In [1]:
import json
import boto3
import pandas as pd

BUCKET = "conrad-mortgage-capstone-2026"

session = boto3.Session(profile_name="capstone")
s3 = session.client("s3")

# Single file inspection
I opened a single file and runs the inspection. I used the UNRATE (unemployment rate) to see how FRED structure its data

In [2]:
key = "bronze/fred/series_id=UNRATE/ingest_date=2026-11-04/observations.json"
data = json.loads(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read())

print("Top level fields:", list(data.keys()))
print("Count:", data["count"])
print("First 4 observations:")
data["observations"][:4]

Top level fields: ['realtime_start', 'realtime_end', 'observation_start', 'observation_end', 'units', 'output_type', 'file_type', 'order_by', 'sort_order', 'count', 'offset', 'limit', 'observations']
Count: 321
First 4 observations:


[{'realtime_start': '2026-10-22',
  'realtime_end': '2026-10-22',
  'date': '2000-01-01',
  'value': '4.0'},
 {'realtime_start': '2026-10-22',
  'realtime_end': '2026-10-22',
  'date': '2000-02-01',
  'value': '4.1'},
 {'realtime_start': '2026-10-22',
  'realtime_end': '2026-10-22',
  'date': '2000-03-01',
  'value': '4.0'},
 {'realtime_start': '2026-10-22',
  'realtime_end': '2026-10-22',
  'date': '2000-04-01',
  'value': '3.8'}]

The file has 321 observations
The series ID (UNRATE) does not appear anywhere in the file
Each observation has a date and a value, plus two realtime fields we do not need
The values are in quotes, meaning they are stored as text and we need to change that. 

In [3]:
# Convert the observations list into a table, preview the first 5 rows,
# and check what data type pandas assigned to each column
df = pd.DataFrame(data["observations"])
print(df.head())
print()
print(df.dtypes)

  realtime_start realtime_end        date value
0     2026-10-22   2026-10-22  2000-01-01   4.0
1     2026-10-22   2026-10-22  2000-02-01   4.1
2     2026-10-22   2026-10-22  2000-03-01   4.0
3     2026-10-22   2026-10-22  2000-04-01   3.8
4     2026-10-22   2026-10-22  2000-05-01   4.0

realtime_start    str
realtime_end      str
date              str
value             str
dtype: object


from the output, we can see that all the columns are stored as text. Value must be converted to a number and date must be changed to a date format

## Checking All 109 Series

Instead of opening every file, I wrote small functions to scan and summarize all 109 at once.

In [4]:
# Collect the path of every FRED data file in Bronze

def list_fred_files():
    pages = s3.get_paginator("list_objects_v2").paginate(Bucket=BUCKET, Prefix="bronze/fred/")
    paths = [f["Key"] for page in pages for f in page["Contents"]]
    return [p for p in paths if p.endswith("observations.json")]

all_files = list_fred_files()
print("Files found:", len(all_files))

Files found: 109


In [5]:
# Get the series ID from the folder path (/series_id=TXUR/ to "TXUR")
def get_series_id(file_path):
    return file_path.split("series_id=")[1].split("/")[0]

# Download one file and return its list of observations
def load_observations(file_path):
    raw = s3.get_object(Bucket=BUCKET, Key=file_path)["Body"].read()
    return json.loads(raw)["observations"]

# Count values FRED marked as missing with "."
def count_missing(observations):
    return sum(1 for item in observations if item["value"] == ".")

# Test each one on the first file
test_path = all_files[0]
test_obs = load_observations(test_path)
print(get_series_id(test_path), len(test_obs), count_missing(test_obs))

AKSTHPI 106 0


In [6]:
# Combine the helpers into a one row summary of a series
def summarize_file(file_path):
    obs = load_observations(file_path)
    return {
        "series_id": get_series_id(file_path),
        "rows": len(obs),
        "first_date": obs[0]["date"],
        "last_date": obs[-1]["date"],
        "missing": count_missing(obs),
        "fields": list(obs[0].keys()),
    }

summarize_file(all_files[0])

{'series_id': 'AKSTHPI',
 'rows': 106,
 'first_date': '2000-01-01',
 'last_date': '2026-04-01',
 'missing': 0,
 'fields': ['realtime_start', 'realtime_end', 'date', 'value']}

In [7]:
# Summarize every file and put the results in a table
profile = pd.DataFrame([summarize_file(p) for p in all_files])
profile = profile.sort_values("series_id")
profile.head(10)

,series_id,rows,first_date,last_date,missing,fields
0,AKSTHPI,106,2000-01-01,2026-04-01,0,"[realtime_start, realtime_end, date, value]"
1,AKUR,321,2000-01-01,2026-09-01,1,"[realtime_start, realtime_end, date, value]"
2,ALSTHPI,106,2000-01-01,2026-04-01,0,"[realtime_start, realtime_end, date, value]"
3,ALUR,321,2000-01-01,2026-09-01,1,"[realtime_start, realtime_end, date, value]"
4,ARSTHPI,106,2000-01-01,2026-04-01,0,"[realtime_start, realtime_end, date, value]"
5,ARUR,321,2000-01-01,2026-09-01,1,"[realtime_start, realtime_end, date, value]"
6,AZSTHPI,106,2000-01-01,2026-04-01,0,"[realtime_start, realtime_end, date, value]"
7,AZUR,321,2000-01-01,2026-09-01,1,"[realtime_start, realtime_end, date, value]"
8,CASTHPI,106,2000-01-01,2026-04-01,0,"[realtime_start, realtime_end, date, value]"
9,CAUR,321,2000-01-01,2026-09-01,1,"[realtime_start, realtime_end, date, value]"


In [8]:
# To see if i got all 109 series
print("Series:", len(profile))

# To check if all the fils have same field
print(profile["fields"].astype(str).value_counts())

# which series have missing values
print(profile[profile["missing"] > 0]["series_id"].tolist())

Series: 109
fields
['realtime_start', 'realtime_end', 'date', 'value']    109
Name: count, dtype: int64
['AKUR', 'ALUR', 'ARUR', 'AZUR', 'CAUR', 'COUR', 'CPIAUCSL', 'CTUR', 'DCUR', 'DEUR', 'FLUR', 'GAUR', 'HIUR', 'IAUR', 'IDUR', 'ILUR', 'INUR', 'KSUR', 'KYUR', 'LAUR', 'MAUR', 'MDUR', 'MEUR', 'MIUR', 'MNUR', 'MOUR', 'MSUR', 'MTUR', 'NCUR', 'NDUR', 'NEUR', 'NHUR', 'NJUR', 'NMUR', 'NVUR', 'NYUR', 'OHUR', 'OKUR', 'ORUR', 'PAUR', 'RIUR', 'SCUR', 'SDUR', 'TNUR', 'TXUR', 'UNRATE', 'UTUR', 'VAUR', 'VTUR', 'WAUR', 'WIUR', 'WVUR', 'WYUR']


In [9]:
# Show the date of every missing value in one series
# (checked CAUR, UNRATE, and CPIAUCSL. all missing 2025-10-01)
obs = load_observations([p for p in all_files if get_series_id(p) == "CPIAUCSL"][0])
print([item["date"] for item in obs if item["value"] == "."])

['2025-10-01']


## Findings: Missing Values

- 53 of 109 series have missing values, and every one is the same single month, October 2025, marked by FRED as `"."`.
- Affected: all 51 state unemployment series, national unemployment (UNRATE), and consumer prices (CPIAUCSL).
- Not affected: house prices, mortgage rates, federal funds rate, GDP, and the recession indicator.

 All affected series are published by the Bureau of Labor Statistics, which canceled its October 2025 employment and CPI reports because the federal government shutdown prevented data collection. The gaps reflect a real world event, not data errors.

My decision is to keep these values as missing (NULL) with `is_missing = true`. Silver records what the source published. Any filling will be a documented decision in the Gold layer if needed.

This has low impact on the project as  economic conditions are joined to loans at origination, and the modeling period (2005 - 2022 vintages) falls well before October 2025.

## Cleaning Plan for the Silver Layer

After checking all 109 FRED files, I found that they all have the same structure. This means one script can clean all of them. Here is what needs to be fixed and how:

1) The data points are packed inside a list called "observations." I will unpack the list so each month (or week or quarter) becomes its own row.
2) The series name (for example, TXUR) is not inside the file. It only appears in the folder name. I will take it from the folder path and add it as a column.
3) The values are saved as text instead of numbers. I will convert them to numbers so they can be used in calculations.
4) Missing values are written as a period ("."). This happens in 53 series, all in October 2025, because of the government shutdown. I will change these to empty (null) values and add a column that marks them as missing.
5) The dates are also saved as text. I will convert them to real date format.
6) Some fields, like realtime_start and realtime_end, are not needed for this project. I will remove them.
7) The series are recorded at different times. Some weekly, some monthly, and some every quarter. I will add a column that shows how often each series is recorded.
8) The series codes are hard to understand on their own. I will add simple labels showing what each series measures (for example, unemployment rate), whether it is national or state level, and which state it belongs to.
9) If the data is downloaded more than once, there will be repeated copies. I will keep only the most recent download of each series.
10) The data is spread across 109 separate files. I will combine them into one table.

Before saving the cleaned data, the script will check that there are no duplicate rows, that every series is recognized, that every date was converted correctly, and that the table is not empty. If any check fails, the script will stop so that bad data is not saved.